In [2]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from app.predictor import MatchPredictor

In [3]:
BASE_DIR = Path.cwd()
MODELS_DIR = BASE_DIR / "models"

model = joblib.load(MODELS_DIR / "best_model.pkl")
feature_columns = joblib.load(MODELS_DIR / "feature_columns.pkl")
team_df = joblib.load(MODELS_DIR / "team_df.pkl")
df_with_elo = joblib.load(MODELS_DIR / "df_with_elo.pkl")

print(type(model))
print(len(feature_columns))
print(team_df.shape)
print(df_with_elo.shape)

<class 'catboost.core.CatBoostClassifier'>
27
(12702, 51)
(6351, 27)


In [4]:
predictor = MatchPredictor(
    model=model,
    feature_columns=feature_columns,
    team_df_model=team_df,
    df_with_elo=df_with_elo
)

In [5]:
result = predictor.predict(
    home_team="arsenal",
    away_team="chelsea",
    match_date="2025-05-10"
)

result

{'home_team': 'arsenal',
 'away_team': 'chelsea',
 'match_date': '2025-05-10',
 'prediction': 'Home Win',
 'probabilities': {'Home Win': 0.5511876869333604,
  'Draw': 0.27494458478870343,
  'Away Win': 0.17386772827793612}}

In [6]:
home_team = "arsenal"
away_team = "chelsea"
match_date = pd.Timestamp("2025-05-10")

home_team_std = home_team.strip().lower()
away_team_std = away_team.strip().lower()

home_hist = (
    team_df[
        (team_df["team"] == home_team_std) &
        (team_df["date"] < match_date)
    ]
    .sort_values("date")
)

away_hist = (
    team_df[
        (team_df["team"] == away_team_std) &
        (team_df["date"] < match_date)
    ]
    .sort_values("date")
)

home_latest = home_hist.iloc[-1]
away_latest = away_hist.iloc[-1]

home_elo_hist_home = df_with_elo[
    (df_with_elo["hometeam"] == home_team_std) &
    (df_with_elo["date"] < match_date)
][["date", "home_elo_pre"]].rename(columns={"home_elo_pre": "elo_pre"})

home_elo_hist_away = df_with_elo[
    (df_with_elo["awayteam"] == home_team_std) &
    (df_with_elo["date"] < match_date)
][["date", "away_elo_pre"]].rename(columns={"away_elo_pre": "elo_pre"})

away_elo_hist_home = df_with_elo[
    (df_with_elo["hometeam"] == away_team_std) &
    (df_with_elo["date"] < match_date)
][["date", "home_elo_pre"]].rename(columns={"home_elo_pre": "elo_pre"})

away_elo_hist_away = df_with_elo[
    (df_with_elo["awayteam"] == away_team_std) &
    (df_with_elo["date"] < match_date)
][["date", "away_elo_pre"]].rename(columns={"away_elo_pre": "elo_pre"})

home_elo_hist = pd.concat([home_elo_hist_home, home_elo_hist_away]).sort_values("date")
away_elo_hist = pd.concat([away_elo_hist_home, away_elo_hist_away]).sort_values("date")

home_elo = float(home_elo_hist.iloc[-1]["elo_pre"])
away_elo = float(away_elo_hist.iloc[-1]["elo_pre"])

feature_row = {}

for col in feature_columns:
    if col == "elo_diff_pre":
        feature_row[col] = float(home_elo - away_elo)
    elif col.startswith("diff_"):
        base_col = col.replace("diff_", "", 1)
        feature_row[col] = float(home_latest[base_col] - away_latest[base_col])

X_match = pd.DataFrame([feature_row])[feature_columns]

print(X_match.shape)
print(X_match.dtypes.head())
X_match.head()

(1, 27)
elo_diff_pre         float64
diff_points_lag_1    float64
diff_gf_lag_1        float64
diff_ga_lag_1        float64
diff_points_lag_2    float64
dtype: object


,elo_diff_pre,diff_points_lag_1,diff_gf_lag_1,diff_ga_lag_1,diff_points_lag_2,diff_gf_lag_2,diff_ga_lag_2,diff_points_lag_3,diff_gf_lag_3,diff_ga_lag_3,...,diff_gf_roll_mean_5,diff_ga_roll_mean_5,diff_points_roll_mean_10,diff_gf_roll_mean_10,diff_ga_roll_mean_10,diff_win_rate_3,diff_win_rate_5,diff_win_rate_10,diff_points_ewm_5,diff_gd_ewm_5
0,83.560813,-2.0,1.0,2.0,0.0,2.0,-1.0,0.0,-1.0,-1.0,...,0.8,0.4,0.0,0.2,-0.2,-0.333333,-0.2,-0.1,-0.592972,0.387544


In [8]:
pred_proba_raw = model.predict_proba(X_match)
pred_class_raw = model.predict(X_match)

print("pred_proba_raw:", pred_proba_raw)
print("pred_class_raw:", pred_class_raw)
print("type(pred_proba_raw):", type(pred_proba_raw))
print("type(pred_class_raw):", type(pred_class_raw))

print("np.asarray(pred_proba_raw):", np.asarray(pred_proba_raw))
print("np.asarray(pred_class_raw):", np.asarray(pred_class_raw))

print("pred_proba shape:", np.asarray(pred_proba_raw).shape)
print("pred_class shape:", np.asarray(pred_class_raw).shape)

pred_proba_raw: [[0.55118769 0.27494458 0.17386773]]
pred_class_raw: [[0]]
type(pred_proba_raw): <class 'numpy.ndarray'>
type(pred_class_raw): <class 'numpy.ndarray'>
np.asarray(pred_proba_raw): [[0.55118769 0.27494458 0.17386773]]
np.asarray(pred_class_raw): [[0]]
pred_proba shape: (1, 3)
pred_class shape: (1, 1)


In [9]:
pred_proba = np.asarray(pred_proba_raw).reshape(-1)
pred_class_arr = np.asarray(pred_class_raw).reshape(-1)

print("flattened pred_proba:", pred_proba)
print("flattened pred_class_arr:", pred_class_arr)
print("first pred class:", pred_class_arr[0])

flattened pred_proba: [0.55118769 0.27494458 0.17386773]
flattened pred_class_arr: [0]
first pred class: 0


In [10]:
print("Model type:", type(model))

if hasattr(model, "classes_"):
    print("Classes:", model.classes_)
else:
    print("Model has no classes_ attribute")

Model type: <class 'catboost.core.CatBoostClassifier'>
Classes: [0 1 2]
